# PPO for LunarLander-v2 | Experiment Report

**Hugging Face Deep Reinforcement Learning Course — Unit 1**  
**Algorithm:** Proximal Policy Optimization (PPO) · **Environment:** LunarLander-v2 · **Framework:** Stable-Baselines3

## Project summary

This notebook documents a completed reinforcement learning experiment in which I trained a PPO agent to land a spacecraft in the discrete-action LunarLander environment. I configured eight parallel training environments, trained the policy for **1,000,000 timesteps**, evaluated it over **20 episodes**, and published the resulting model on the Hugging Face Hub.

**Outcome:** The local evaluation produced a mean reward of **271.67 ± 18.98** (mean ± standard deviation), corresponding to a score of **252.69**. The official Hugging Face progress checker reported **235.88**, above the **200-point** passing threshold. These are results from the completed experiment, not outputs regenerated in this edited notebook.

**Published model:** [Wjaniak2310/ppo-LunarLander-v2](https://huggingface.co/Wjaniak2310/ppo-LunarLander-v2)

> **Attribution:** This experiment was completed as part of the [Hugging Face Deep RL Course, Unit 1 practical assignment](https://huggingface.co/learn/deep-rl-course/en/unit1/hands-on). The exercise and original notebook were provided by the course; the configuration, run, evaluation, and documentation here describe my completed work.

## 1. Experimental environment

LunarLander-v2 is a simulated control task in which the agent must land a spacecraft safely. Each observation contains **eight values**: horizontal and vertical position, horizontal and vertical velocity, angle, angular velocity, and two landing-leg contact indicators. The discrete action space contains **four actions**: no thrust, left orientation engine, main engine, and right orientation engine.

The experiment was conducted in **Google Colab** using **Python**, **Gymnasium**, and **Stable-Baselines3**. The dependency versions below correspond to the working LunarLander-v2 setup. They are retained to document the software environment and support reproduction.

In [ ]:
!apt-get update -qq
!apt-get install -y swig build-essential python3-dev
%pip install --no-cache-dir "gymnasium[box2d]==0.29.1" "stable-baselines3==2.3.2" "huggingface-sb3"

In [ ]:
import gymnasium as gym

ENV_ID = "LunarLander-v2"
env = gym.make(ENV_ID)
observation, info = env.reset(seed=42)
print("Gymnasium version:", gym.__version__)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial observation shape:", observation.shape)
env.close()

## 2. PPO methodology

I used **Proximal Policy Optimization (PPO)** with an `MlpPolicy` because the environment provides a compact numerical observation vector. Eight parallel instances of LunarLander collected experience during training. PPO uses a clipped objective to limit the magnitude of policy updates.

| Hyperparameter | Experiment setting |
|---|---:|
| Parallel environments | 8 |
| Training timesteps | 1,000,000 |
| Rollout steps per environment | 1,024 |
| Minibatch size | 64 |
| Optimization epochs per update | 4 |
| Learning rate | 0.0003 |
| Discount factor (`gamma`) | 0.999 |
| GAE parameter (`gae_lambda`) | 0.98 |
| Entropy coefficient | 0.01 |
| Policy | `MlpPolicy` |
| Compute device | CPU |

The original training run did not specify a random seed. Consequently, rerunning the experiment may yield different trained weights and evaluation results.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Create 8 parallel environments
train_env = make_vec_env("LunarLander-v2", n_envs=8)

# Configure PPO
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    learning_rate=3e-4,
    verbose=1,
    device="cpu"
)

print("PPO model created successfully!")

## 3. Training

The agent was trained for **1,000,000 environment timesteps**. After training, the policy was saved as `ppo-LunarLander-v2.zip`. The code below is the training procedure used for the completed experiment; running it again creates a **new training run**, rather than reproducing the exact previously published weights.

In [ ]:
model.learn(total_timesteps=1_000_000)

model.save("ppo-LunarLander-v2")

## 4. Evaluation protocol

I evaluated the trained policy on a separate `Monitor`-wrapped LunarLander-v2 environment for **20 episodes**, using deterministic action selection. The local score was calculated as:

**Score = Mean episode reward − Standard deviation**

The code below performs this evaluation for the model held in memory. Results from a fresh run may differ from the historical results reported afterward.

In [ ]:
import gymnasium as gym

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

eval_env = Monitor(gym.make("LunarLander-v2"))

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True
)

score = mean_reward - std_reward

print(f"Mean reward: {mean_reward:.2f}")
print(f"Standard deviation: {std_reward:.2f}")
print(f"Score: {score:.2f}")

eval_env.close()

### Results of the completed experiment

| Metric | Recorded result |
|---|---:|
| Evaluation episodes (local) | 20 |
| Mean episode reward | **271.67** |
| Reward standard deviation | **18.98** |
| Local score (mean − std) | **252.69** |
| Official Hugging Face score | **235.88** |
| Passing threshold | **200** |
| Official assignment status | **Passed** |

Both the local and official scores exceeded the course threshold. The official score was produced by the Hugging Face checker and **is not the same evaluation run** as the local 20-episode measurement.

**Interpretation.** The local mean reward indicates that the policy performed well across the sampled episodes, while the standard deviation captures variation between landings. The difference between local and official scores is consistent with evaluations using different episodes and/or evaluation settings; it should not be interpreted as a change in training performance without a controlled comparison.

**Limitations.** This was a single documented training run, without multiple random seeds or a baseline-algorithm comparison. The results therefore demonstrate completion of the course task, not statistical superiority over alternative methods. Training curves were not preserved in the available notebook, so none are reconstructed or fabricated here.

## 5. Published model

The trained policy was published to the [Hugging Face Hub](https://huggingface.co/Wjaniak2310/ppo-LunarLander-v2), where the model artifact can be accessed independently of this notebook.

The following cells preserve the **optional publication procedure** used with the course tooling. They are not required to inspect this report or understand the results. Running them requires Hugging Face authentication and can update the existing model repository.

In [ ]:
from huggingface_hub import login, HfApi

# Enter your write-enabled token only in the secure prompt.
login()
print("Authenticated as:", HfApi().whoami()["name"])

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from huggingface_sb3 import package_to_hub
from stable_baselines3.common.monitor import Monitor

repo_id = "Wjaniak2310/ppo-LunarLander-v2"
model_name = "ppo-LunarLander-v2"

video_eval_env = DummyVecEnv([
    lambda: Monitor(gym.make(ENV_ID, render_mode="rgb_array"))
])

try:
    package_to_hub(
        model=model,
        model_name=model_name,
        model_architecture="PPO",
        env_id=ENV_ID,
        eval_env=video_eval_env,
        repo_id=repo_id,
        commit_message="Publish PPO LunarLander-v2 agent",
    )
finally:
    video_eval_env.close()

## 6. Discussion and conclusion

This experiment demonstrated an end-to-end reinforcement learning workflow: inspecting a control environment, configuring an MLP-based PPO policy, collecting experience with parallel environments, training for one million timesteps, evaluating the learned policy, and publishing the model.

The agent achieved a **252.69 local score** and a **235.88 official score**, passing the Unit 1 requirement of 200. A logical extension would be to repeat training across several random seeds, log reward curves, and compare PPO hyperparameter settings under a consistent evaluation protocol.

### References

- [Hugging Face Deep Reinforcement Learning Course — Unit 1 hands-on](https://huggingface.co/learn/deep-rl-course/en/unit1/hands-on)
- [Stable-Baselines3 documentation](https://stable-baselines3.readthedocs.io/)
- [Gymnasium LunarLander documentation](https://gymnasium.farama.org/environments/box2d/lunar_lander/)
- [Published PPO model](https://huggingface.co/Wjaniak2310/ppo-LunarLander-v2)